In [ ]:
import cv2
import torch
import numpy as np
from PIL import Image
from torchvision import transforms
import torchvision.models as models
import torch.nn as nn
from ultralytics import YOLOWorld

# ==========================================
# 1. Helper Functions
# ==========================================
def calculate_iou(boxA, boxB):
    """Calculates Intersection over Union to determine if bounding boxes overlap."""
    xA, yA = max(boxA[0], boxB[0]), max(boxA[1], boxB[1])
    xB, yB = min(boxA[2], boxB[2]), min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    return interArea / float(boxAArea + boxBArea - interArea + 1e-5)

# Load Model (Same as before) ---
def load_model(model_path, num_classes=3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = models.resnet18(num_classes=365) # Initialize structure
    model.fc = nn.Linear(model.fc.in_features, num_classes) # Fix head
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    return model

def extract_pure_scene_graph(results):
    """Generates raw triplets from YOLO-World detections purely based on spatial relationships."""
    triplets = set()
    print("length of results: ", len(results[0].boxes))
    if len(results) == 0 or len(results[0].boxes) == 0:
        return triplets

    boxes = results[0].boxes.xyxy.cpu().numpy()
    classes = results[0].boxes.cls.cpu().numpy()
    conf = results[0].boxes.conf.cpu().numpy()
    names = results[0].names

    people = []
    objects = []

    print(f"YOLO-World detected {len(boxes)} objects and {len(classes)} classes. Classes: {[names[int(c)] for c in classes]}")
    # Parse nodes
    for i, cls_id in enumerate(classes):
        print(f"Detection {i}: Class ID {cls_id}, Confidence {conf[i]:.2f}")
        raw_label = names[int(cls_id)]
        # Fire/Smoke are harder to detect, so we accept lower confidence
        threshold = 0.10 if raw_label in ["fire", "smoke", "flames"] else 0.25
        
        if conf[i] < threshold: 
            continue
        if conf[i] < 0.3: # Confidence threshold
            continue
            
        label = names[int(cls_id)]
        box = boxes[i]
        print(f"Detected: {label} with confidence {conf[i]:.2f} at {box}")
        
        if label == "person":
            people.append(box)
        else:
            objects.append((label, box))
            
            # Handle state-based nodes detected directly
            if label in ["fire", "smoke"]:
                triplets.add(f"[{label.capitalize()}, is, present]")
            elif label == "person falling":
                triplets.add(f"[Person, is, falling]")

    # Parse edges (Relationships mapped via overlap logic)
    for p_box in people:
        for obj_label, obj_box in objects:
            if calculate_iou(p_box, obj_box) > 0.1:  
                triplets.add(f"[Person, touching/holding, {obj_label.capitalize()}]")
            else:
                triplets.add(f"[Person, near, {obj_label.capitalize()}]")

    return triplets


# ==========================================
# 2. Main Video Processing Function
# ==========================================
def classify_video(video_path, env_model, sgg_model, class_names):
    device = next(env_model.parameters()).device
    
    # Standard Transform for the Environment CNN
    preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return "Error opening video", 0.0, ""

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    # Sample 5 frames evenly spaced
    frame_indices = np.linspace(0, total_frames-1, 5, dtype=int)
    
    accumulated_probs = torch.zeros(len(class_names)).to(device)
    valid_frames = 0
    accumulated_triplets = set() 

    print(f"Scanning '{video_path}' ({total_frames} frames)...")

    for i in range(total_frames):
        ret, frame = cap.read()
        if not ret: break
        
        if i in frame_indices:
            # --- SGG INFERENCE (YOLO-World) ---
            sgg_results = sgg_model.predict(frame, verbose=False)
            # print(f"frame {i} sgg results: {sgg_results}")
            frame_triplets = extract_pure_scene_graph(sgg_results)
            for triplet in frame_triplets:
                accumulated_triplets.add(triplet)

            # --- ENVIRONMENT INFERENCE (CNN) ---
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(frame_rgb)
            
            input_tensor = preprocess(pil_image).unsqueeze(0).to(device)
            
            with torch.no_grad():
                outputs = env_model(input_tensor)
                probs = torch.nn.functional.softmax(outputs[0], dim=0)
                accumulated_probs += probs
                valid_frames += 1

    cap.release()

    if valid_frames == 0: 
        return "No frames read", 0.0, ""
    
    # Calculate average environment probabilities
    avg_probs = accumulated_probs / valid_frames
    confidence, predicted_idx = torch.max(avg_probs, 0)
    final_environment = class_names[predicted_idx.item()]
    
    # Format the final string exactly as the LLM expects it
    graph_string = " | ".join(accumulated_triplets) if accumulated_triplets else "[None]"
    llm_input_string = f"Environment: {final_environment}. Graph: {graph_string}"
    
    return final_environment, confidence.item(), llm_input_string

# ==========================================
# 3. Execution Block
# ==========================================
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. Initialize your specific Places365 Environment Model
    MODEL_PATH = 'places365_environment_model_new.pth'
    CLASSES = ['classroom', 'home', 'office']

    env_classifier = load_model(MODEL_PATH, len(CLASSES)) 

    # 2. Initialize the YOLO-World Perception Layer
    sgg_yolo = YOLOWorld("yolov8s-world.pt")
    # Define the vocabulary you want the SGG to look for
    # sgg_vocab = ["person", "gun", "knife", "fire", "smoke", "person falling", "chair", "desk", "laptop"]
    sgg_vocab = [
    "person", "gun", "knife", 
    "fire", "flames", "smoke", "fireplace", "burning",  # Expanded Fire terms
    "person falling", "chair", "desk", "laptop", "projector"
    ]
    sgg_yolo.set_classes(sgg_vocab)

    # 3. Process the Local Video
    # video_file = 'Data/office-trimmed.mp4' 
    video_file = 'Data/living-room-fireplace.mp4'
    # video_file = 'Data/classroom-sample-1.mp4'

    
    detected_env, conf, final_llm_prompt = classify_video(
        video_path=video_file, 
        env_model=env_classifier, 
        sgg_model=sgg_yolo, 
        class_names=CLASSES
    )

    print("\n--- Final Pipeline Output ---")
    print(f"Environment: {detected_env} (Confidence: {conf:.2f})")
    print(f"LLM Prompt : {final_llm_prompt}")

Scanning 'Data/living-room-fireplace.mp4' (251 frames)...
length of results:  1
YOLO-World detected 1 objects and 1 classes. Classes: ['fireplace']
Detection 0: Class ID 6.0, Confidence 0.29
length of results:  1
YOLO-World detected 1 objects and 1 classes. Classes: ['fireplace']
Detection 0: Class ID 6.0, Confidence 0.36
Detected: fireplace with confidence 0.36 at [     577.14      325.27      747.02      468.97]
length of results:  1
YOLO-World detected 1 objects and 1 classes. Classes: ['fireplace']
Detection 0: Class ID 6.0, Confidence 0.38
Detected: fireplace with confidence 0.38 at [     576.52      325.48      746.57      469.37]
length of results:  1
YOLO-World detected 1 objects and 1 classes. Classes: ['fireplace']
Detection 0: Class ID 6.0, Confidence 0.35
Detected: fireplace with confidence 0.35 at [     582.24      327.79       746.3      468.65]
length of results:  1
YOLO-World detected 1 objects and 1 classes. Classes: ['fireplace']
Detection 0: Class ID 6.0, Confidence 

In [29]:
## Modified the code to generate a pure triplet-based graph string instead of a JSON, as per the LLM training format. 
# The environment classification remains intact, and the SGG now focuses on generating concise triplets that describe the scene in a way that the LLM can easily parse and reason about.
import cv2
import json
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
from torchvision import transforms
import torchvision.models as models
from ultralytics import YOLOWorld

# ==========================================
# 1. Helper Functions
# ==========================================
def calculate_iou(boxA, boxB):
    """Calculates Intersection over Union to determine if bounding boxes overlap."""
    xA, yA = max(boxA[0], boxB[0]), max(boxA[1], boxB[1])
    xB, yB = min(boxA[2], boxB[2]), min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    return interArea / float(boxAArea + boxBArea - interArea + 1e-5)

# Load Model (Same as before) ---
def load_model(model_path, num_classes=3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = models.resnet18(num_classes=365) # Initialize structure
    model.fc = nn.Linear(model.fc.in_features, num_classes) # Fix head
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    return model

def extract_json_scene_graph(sgg_results, environment_label, scene_id):
    """Generates a structured JSON scene graph matching the LLM training format."""
    
    graph_dict = {
        "scene_id": scene_id,
        "environment": environment_label.capitalize(),
        "event": "",  # Left intentionally blank for the LLM to predict
        "nodes": [],
        "edges": []
    }
    
    if sgg_results is None or len(sgg_results) == 0 or len(sgg_results[0].boxes) == 0:
        return json.dumps(graph_dict)

    boxes = sgg_results[0].boxes.xyxy.cpu().numpy()
    classes = sgg_results[0].boxes.cls.cpu().numpy()
    conf = sgg_results[0].boxes.conf.cpu().numpy()
    names = sgg_results[0].names

    node_data = {} 
    node_counter = 1
    
    # --- A. Generate Nodes ---
    for i, cls_id in enumerate(classes):
        if conf[i] < 0.3: 
            continue
            
        raw_label = names[int(cls_id)]
        node_id = f"n{node_counter}"
        
        # Handle states and format the label
        if raw_label == "person falling":
            label = "Person"
            attributes = ["falling"]
        else:
            label = raw_label.capitalize()
            attributes = []

        graph_dict["nodes"].append({
            "id": node_id,
            "label": label,
            "attributes": attributes
        })
        
        node_data[node_id] = {"box": boxes[i], "label": label}
        node_counter += 1

    # --- B. Generate Edges ---
    node_ids = list(node_data.keys())
    for i in range(len(node_ids)):
        for j in range(i + 1, len(node_ids)):
            id1 = node_ids[i]
            id2 = node_ids[j]
            data1 = node_data[id1]
            data2 = node_data[id2]
            
            # Map Person -> Object interactions
            if data1["label"] == "Person" and data2["label"] != "Person":
                person_id, obj_id = id1, id2
            elif data2["label"] == "Person" and data1["label"] != "Person":
                person_id, obj_id = id2, id1
            else:
                continue # Skip Object-to-Object relationships to keep the prompt lean

            # Apply spatial overlap logic to determine relationship type
            iou = calculate_iou(node_data[person_id]["box"], node_data[obj_id]["box"])
            relationship = "touching_or_holding" if iou > 0.1 else "near"
                
            graph_dict["edges"].append({
                "source": person_id,
                "target": obj_id,
                "relationship": relationship
            })

    return json.dumps(graph_dict)

def extract_dynamic_json_scene_graph(sgg_results, environment_label, scene_id):
    """
    Generates a scene graph dynamically based ONLY on what YOLO detects.
    No hardcoded 'if label == fire' logic.
    """
    graph_dict = {
        "scene_id": scene_id,
        "environment": environment_label, # e.g. "Living Room"
        "event": "", 
        "nodes": [],
        "edges": []
    }
    
    if sgg_results is None or len(sgg_results) == 0 or len(sgg_results[0].boxes) == 0:
        return json.dumps(graph_dict)

    boxes = sgg_results[0].boxes.xyxy.cpu().numpy()
    classes = sgg_results[0].boxes.cls.cpu().numpy()
    conf = sgg_results[0].boxes.conf.cpu().numpy()
    names = sgg_results[0].names

    node_data = {} 
    node_counter = 1
    
    # --- A. Dynamic Node Generation ---
    for i, cls_id in enumerate(classes):
        if conf[i] < 0.2: # Single global threshold
            continue
            
        # 1. Get the raw string directly from YOLO (e.g., "person falling", "fire", "gun")
        raw_label = names[int(cls_id)]
        
        # 2. auto-format: "person falling" -> Label: "Person Falling"
        # We don't try to split it into attributes manually. We trust the LLM to read "Person Falling".
        label = raw_label.replace("_", " ").title()
        
        node_id = f"n{node_counter}"
        
        graph_dict["nodes"].append({
            "id": node_id,
            "label": label, 
            "attributes": [] # We leave this empty; the label itself ("Burning Fire") contains the info
        })
        
        node_data[node_id] = {"box": boxes[i], "label": label}
        node_counter += 1

    # --- B. Dynamic Edge Generation ---
    # We map spatial relationships between ALL objects, not just people.
    # The LLM needs to know if "Fire" is near "Curtains" or inside "Fireplace".
    node_ids = list(node_data.keys())
    
    for i in range(len(node_ids)):
        for j in range(i + 1, len(node_ids)):
            id1 = node_ids[i]
            id2 = node_ids[j]
            data1 = node_data[id1]
            data2 = node_data[id2]
            
            iou = calculate_iou(data1["box"], data2["box"])
            
            # Simple spatial logic is generic for ANY object pair
            if iou > 0.05:
                rel = "interacting_with" # Generic overlap
            elif iou > 0.0:
                rel = "adjacent_to"      # Touching borders
            else:
                # Calculate distance if needed, or skip non-overlapping to keep graph small
                continue 

            graph_dict["edges"].append({
                "source": id1,
                "target": id2,
                "relationship": rel
            })

    return json.dumps(graph_dict)
# ==========================================
# 2. Main Video Processing Function
# ==========================================
def classify_video(video_path, env_model, sgg_model, class_names, scene_id=1):
    device = next(env_model.parameters()).device
    
    preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return "Error opening video", 0.0, "{}"

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    # frame_indices = np.linspace(0, total_frames-1, 5, dtype=int) ## Evenly sample 5 frames

    # --- NEW SAMPLING LOGIC ---
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0: fps = 30 # Fallback if metadata is missing

    # We want 5 frames per second. 
    # If FPS is 30, we step every 6 frames.
    sample_interval = int(fps / 5)
    
    # Generate the exact indices we want to look at
    # starting at a slight offset (sample_interval // 2) avoids the very first millisecond which is often black
    frame_indices = list(range(sample_interval // 2, total_frames, sample_interval))
    
    # Quick sanity check: if video is too short, take at least 1 frame
    if not frame_indices:
        frame_indices = [total_frames // 2]
    # ---------------------------
    
    accumulated_probs = torch.zeros(len(class_names)).to(device)
    valid_frames = 0
    
    # Tracking variables for the SGG
    best_sgg_results = None
    max_detections = -1

    print(f"Scanning '{video_path}' ({total_frames} frames)...")

    # Loop through the video frames and then process only the sampled frames for both SGG and Environment classification
    for i in range(total_frames):
        ret, frame = cap.read()
        if not ret: break
        
        if i in frame_indices:
            # --- 1. SGG INFERENCE (YOLO-World) ---
            sgg_results = sgg_model.predict(frame, verbose=False)
            
            # Save the frame that has the highest number of detected objects
            num_detections = len(sgg_results[0].boxes) if len(sgg_results) > 0 else 0
            if num_detections > max_detections:
                max_detections = num_detections
                best_sgg_results = sgg_results 

            # --- 2. ENVIRONMENT INFERENCE (CNN) ---
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(frame_rgb)
            
            input_tensor = preprocess(pil_image).unsqueeze(0).to(device)
            
            with torch.no_grad():
                outputs = env_model(input_tensor)
                probs = torch.nn.functional.softmax(outputs[0], dim=0)
                accumulated_probs += probs
                valid_frames += 1

    cap.release()

    if valid_frames == 0: 
        return "No frames read", 0.0, "{}"
    
    # Average environment probabilities across sampled frames
    avg_probs = accumulated_probs / valid_frames
    confidence, predicted_idx = torch.max(avg_probs, 0)
    final_environment = class_names[predicted_idx.item()]
    
    # Generate the final JSON graph using the most active frame
    # final_json_string = extract_json_scene_graph(
    #     sgg_results=best_sgg_results, 
    #     environment_label=final_environment,
    #     scene_id=scene_id
    # )
    final_json_string = extract_dynamic_json_scene_graph(
        sgg_results=best_sgg_results, 
        environment_label=final_environment,
        scene_id=scene_id
    )
    
    return final_environment, confidence.item(), final_json_string

# ==========================================
# 3. Execution Block
# ==========================================
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. Initialize your specific Places365 Environment Model
    MODEL_PATH = 'places365_environment_model_new.pth'
    CLASSES = ['classroom', 'home', 'office']

    env_classifier = load_model(MODEL_PATH, len(CLASSES))

    # 2. Initialize the YOLO-World Perception Layer
    sgg_yolo = YOLOWorld("yolov8s-world.pt")
    # sgg_vocab = ["person", "gun", "knife", "fire", "smoke", "person falling", "chair", "desk", "laptop", "projector"]
    sgg_vocab = [
    "person", "gun", "knife", 
    "fire", "flames", "smoke", "fireplace", "burning",  # Expanded Fire terms
    "person falling", "chair", "desk", "laptop", "projector"
    ]
    sgg_yolo.set_classes(sgg_vocab)

    # 3. Process the Local Video
    # video_file = 'Data/office-trimmed.mp4'
    video_file = 'Data/living-room-fireplace.mp4'
    # video_file = 'Data/person_Holding_a Gun indoors.mp4'
    # video_file = 'Data/classroom-sample-1.mp4'
    # video_file = 'Data/living room on fire 2.mp4'
    # video_file = 'Data/living-room-fire-1.mp4'
    
    detected_env, conf, final_llm_json = classify_video(
        video_path=video_file, 
        env_model=env_classifier, 
        sgg_model=sgg_yolo, 
        class_names=CLASSES,
        scene_id=1
    )

    print("\n--- Final Pipeline Output ---")
    print(f"Predicted Environment: {detected_env.capitalize()} (Confidence: {conf:.2f})")
    print("\n--- LLM Input JSON ---")
    print(final_llm_json)

Scanning 'Data/living-room-fireplace.mp4' (251 frames)...

--- Final Pipeline Output ---
Predicted Environment: Home (Confidence: 1.00)

--- LLM Input JSON ---
{"scene_id": 1, "environment": "home", "event": "", "nodes": [{"id": "n1", "label": "Fireplace", "attributes": []}], "edges": []}
